<a href="https://colab.research.google.com/github/anandharidas/KernelProgramming/blob/main/KernelOptimization_Part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
# Optimization Part 1
# Matrix Multiplcation Example
# Kernel 1 - Naive
# Kernel 2 - Row Coallesced
# Kernel 3 - Shared Memory Chunked
# Kernel 4 - 1D Tiling



In [14]:
%%writefile kernel-1-naive.py

"""
  load from mata -> uncoalsced
  load from matb -> coalcsced
  store to matc -> uncoalsced
"""

# Kernel 1
from numba import cuda, float32
import cupy as cp
import numpy as np

ceil_div = lambda a,b : (a + b - 1) // b

@cuda.jit
def matmul (matA , matB , matC, m, n, k):
  row = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
  col = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
  if row < m and col < n:
    matC[row, col] = float32(0.0)
    for i in range(k):
      matC[row, col] += matA[row, i] * matB[i, col]


m = 1000
k = 1200
n =  800

matA = cp.ones((m, k), dtype=cp.float32) * 2.0
matB = cp.ones((k, n), dtype=cp.float32) * 3.0
matC = cp.zeros((m, n), dtype=cp.float32)

threads_per_block = (32, 32)
blocks_per_grid = (ceil_div(m, threads_per_block[0]), ceil_div(n, threads_per_block[1]))

matmul[blocks_per_grid, threads_per_block](matA, matB, matC, m, n, k)

# cudaMemcpy2D / cudamemcpy in C++
matA = cp.asnumpy(matA)
matB = cp.asnumpy(matB)
matC = cp.asnumpy(matC)

np.testing.assert_allclose(matC, matA @ matB, rtol=1e-5,atol=1e-5)

#

Overwriting kernel-1-naive.py


In [15]:
!ncu --set full -f -o kernel-1-naive-rep python3 kernel-1-naive.py

==PROF== Connected to process 19412 (/usr/bin/python3.13)
==PROF== Profiling "cupy_fill" - 0: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_multiply__float32_float_..." - 1: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_fill" - 2: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_multiply__float32_float_..." - 3: 0%....50%....100% - 33 passes
==PROF== Profiling "matmul[abi:v1,cw51cXTLSUwv1sD..." - 4: 0%....50%....100% - 33 passes
==PROF== Disconnected from process 19412
==PROF== Report: /content/kernel-1-naive-rep.ncu-rep


In [16]:
%%writefile kernel-2-coallesced.py

"""
  load from mata -> coalsced
  load from matb -> uncoalcsced
  store to matc -> coalsced
"""

# Kernel 2
from numba import cuda, float32
import cupy as cp
import numpy as np

ceil_div = lambda a,b : (a + b - 1) // b

@cuda.jit
def matmul (matA , matB , matC, m, n, k):
  # NOTICE the SWAP col v/s row
  col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
  row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
  if row < m and col < n:
    matC[row, col] = float32(0.0)
    for i in range(k):
      matC[row, col] += matA[row, i] * matB[i, col]


m = 1000
k = 1200
n =  800

matA = cp.ones((m, k), dtype=cp.float32) * 2.0
matB = cp.ones((k, n), dtype=cp.float32) * 3.0
matC = cp.zeros((m, n), dtype=cp.float32)

threads_per_block = (32, 32)
# NOTICE the SWAP n v/s m = col v/s row
blocks_per_grid =  (
    ceil_div(n, threads_per_block[0]),
    ceil_div(m, threads_per_block[1])
)

matmul[blocks_per_grid, threads_per_block](matA, matB, matC, m, n, k)
cuda.synchronize()

# cudaMemcpy2D / cudamemcpy in C++
matA = cp.asnumpy(matA)
matB = cp.asnumpy(matB)
matC = cp.asnumpy(matC)

np.testing.assert_allclose(matC, matA @ matB, rtol=1e-5,atol=1e-5)

#

Overwriting kernel-2-coallesced.py


In [17]:
!ncu --set full -f -o kernel-2-coallesced-rep python3 kernel-2-coallesced.py

==PROF== Connected to process 19836 (/usr/bin/python3.13)
==PROF== Profiling "cupy_fill" - 0: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_multiply__float32_float_..." - 1: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_fill" - 2: 0%....50%....100% - 33 passes
==PROF== Profiling "cupy_multiply__float32_float_..." - 3: 0%....50%....100% - 33 passes
==PROF== Profiling "matmul[abi:v1,cw51cXTLSUwv1sD..." - 4: 0%....50%....100% - 33 passes
==PROF== Disconnected from process 19836
==PROF== Report: /content/kernel-2-coallesced-rep.ncu-rep
